# Loan Approval Prediction Model

By Jompe Emmanuel Ayomiposi, Level 3 Artificial Intelligence Project

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Import Required Libraries
# We import all libraries needed for data processing, visualization,
# classification modelling, and model persistence.
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd                          # Data manipulation and analysis
import numpy as np                           # Numerical computations
import matplotlib.pyplot as plt              # Plotting and visualization
import seaborn as sns                        # Statistical data visualization
import warnings                              # Suppress unwanted warnings

from sklearn.model_selection import train_test_split        # Split data into train/test sets
from sklearn.ensemble import RandomForestClassifier         # Main classification model
from sklearn.metrics import (
    accuracy_score,          # Overall correct prediction rate
    classification_report,   # Precision, recall, F1-score per class
    confusion_matrix         # True/False Positives & Negatives matrix
)
import joblib                                               # Save/load trained model

warnings.filterwarnings('ignore')    # Hide deprecation warnings for cleaner output
%matplotlib inline                   # Display plots inline within the notebook

print("✅ Libraries imported successfully")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Load the Dataset
# We load the loan approval CSV and inspect its structure, shape, and types.
# ─────────────────────────────────────────────────────────────────────────────

# Load the dataset from the same directory as this notebook
df = pd.read_csv('loan_approval.csv')

# Display first 5 rows to understand the structure
print("📋 First 5 rows of the dataset:")
print(df.head())

print(f"\n📐 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n📊 Column Data Types:")
print(df.dtypes)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Exploratory Data Analysis (EDA) — Statistics
# We examine the statistical distribution of numeric features and the
# overall loan approval rate to understand the balance of the dataset.
# ─────────────────────────────────────────────────────────────────────────────

# ── 3.1 Statistical Summary ──────────────────────────────────────────────────
print("📈 Statistical Summary:")
print(df.describe())

# ── 3.2 Missing Values Check ─────────────────────────────────────────────────
print("\n🔍 Missing Values per Column:")
print(df.isnull().sum())

# ── 3.3 Loan Approval Distribution ───────────────────────────────────────────
approval_counts = df['loan_approved'].value_counts()
approval_pct = df['loan_approved'].value_counts(normalize=True) * 100
print("\n📊 Loan Approval Distribution:")
print(f"   Approved (True)  : {approval_counts.get(True, 0):>5} records ({approval_pct.get(True, 0):.1f}%)")
print(f"   Denied   (False) : {approval_counts.get(False, 0):>5} records ({approval_pct.get(False, 0):.1f}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Visualizations (EDA Plots)
# We visualize the distribution of key features and their relationship
# to loan approval outcomes.
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Loan Approval Dataset — Exploratory Data Analysis', fontsize=16, fontweight='bold')

# Plot 1: Loan Approval Rate (Pie Chart)
approval_counts = df['loan_approved'].value_counts()
axes[0, 0].pie(
    approval_counts.values,
    labels=['Denied', 'Approved'],
    colors=['#e74c3c', '#2ecc71'],
    autopct='%1.1f%%',
    startangle=90
)
axes[0, 0].set_title('Loan Approval Rate')

# Plot 2: Credit Score Distribution by Approval Status
df[df['loan_approved'] == True]['credit_score'].hist(ax=axes[0, 1], bins=30, alpha=0.7, color='#2ecc71', label='Approved')
df[df['loan_approved'] == False]['credit_score'].hist(ax=axes[0, 1], bins=30, alpha=0.7, color='#e74c3c', label='Denied')
axes[0, 1].set_title('Credit Score Distribution')
axes[0, 1].set_xlabel('Credit Score')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# Plot 3: Income Distribution by Approval Status
df[df['loan_approved'] == True]['income'].hist(ax=axes[0, 2], bins=30, alpha=0.7, color='#2ecc71', label='Approved')
df[df['loan_approved'] == False]['income'].hist(ax=axes[0, 2], bins=30, alpha=0.7, color='#e74c3c', label='Denied')
axes[0, 2].set_title('Income Distribution')
axes[0, 2].set_xlabel('Income ($)')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].legend()

# Plot 4: Points Distribution
df[df['loan_approved'] == True]['points'].hist(ax=axes[1, 0], bins=20, alpha=0.7, color='#2ecc71', label='Approved')
df[df['loan_approved'] == False]['points'].hist(ax=axes[1, 0], bins=20, alpha=0.7, color='#e74c3c', label='Denied')
axes[1, 0].set_title('Points Distribution')
axes[1, 0].set_xlabel('Points')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()

# Plot 5: Loan Amount Distribution
df[df['loan_approved'] == True]['loan_amount'].hist(ax=axes[1, 1], bins=30, alpha=0.7, color='#2ecc71', label='Approved')
df[df['loan_approved'] == False]['loan_amount'].hist(ax=axes[1, 1], bins=30, alpha=0.7, color='#e74c3c', label='Denied')
axes[1, 1].set_title('Loan Amount Distribution')
axes[1, 1].set_xlabel('Loan Amount ($)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()

# Plot 6: Correlation Heatmap (numeric features only)
numeric_df = df.select_dtypes(include=np.number)
numeric_df['loan_approved'] = df['loan_approved'].astype(int)
corr = numeric_df.corr()
sns.heatmap(corr, ax=axes[1, 2], annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
axes[1, 2].set_title('Feature Correlation Heatmap')

plt.tight_layout()
plt.savefig('eda_loan_approver.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA plots saved as eda_loan_approver.png")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Data Cleaning & Preprocessing
# We prepare the dataset by:
#   • Dropping columns that are NOT predictive (name, city)
#   • Converting the boolean target to integer (True→1, False→0)
#   • Checking for and handling any remaining null values
# ─────────────────────────────────────────────────────────────────────────────

# ── 5.1 Drop non-predictive columns ──────────────────────────────────────────
# 'name' is a unique identifier with no predictive value
# 'city' has too many unique values and no ordinal meaning for this model
columns_to_drop = ['name', 'city']
df.drop(columns=columns_to_drop, inplace=True)
print(f"🗑️  Dropped columns: {columns_to_drop}")

# ── 5.2 Convert boolean target to integer ─────────────────────────────────────
# sklearn classifiers expect numeric labels: True→1, False→0
df['loan_approved'] = df['loan_approved'].astype(int)
print("🔄 Converted 'loan_approved' boolean → integer (True=1, False=0)")

# ── 5.3 Check and handle missing values ───────────────────────────────────────
null_count = df.isnull().sum().sum()
if null_count > 0:
    df.fillna(df.median(numeric_only=True), inplace=True)
    print(f"✅ Filled {null_count} missing values with column medians")
else:
    print("✅ No missing values found — dataset is clean")

print(f"\n📋 Cleaned dataset shape: {df.shape}")
print(df.head())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Feature Engineering & Dataset Split
# We define our feature matrix X and target vector y, then split the data
# into training (80%) and testing (20%) sets.
#
# Features (X) — all numeric columns except target:
#   income, credit_score, loan_amount, years_employed, points
#
# Target (y):
#   loan_approved  →  1 (Approved) or 0 (Denied)
# ─────────────────────────────────────────────────────────────────────────────

# Define features and target
X = df.drop(columns=['loan_approved'])   # All columns except the target
y = df['loan_approved']                   # Target: 1=Approved, 0=Denied

print(f"📋 Features: {list(X.columns)}")
print(f"📐 Feature matrix shape: {X.shape}")
print(f"📐 Target vector shape : {y.shape}")

# Split into training and testing sets (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% held out for unbiased evaluation
    random_state=42,     # Fixed seed for reproducibility
    stratify=y           # Preserve class balance in both splits
)

print("\n✅ Data split complete:")
print(f"   Training samples : {X_train.shape[0]}")
print(f"   Testing samples  : {X_test.shape[0]}")
print(f"   Approval rate (train): {y_train.mean()*100:.1f}%")
print(f"   Approval rate (test) : {y_test.mean()*100:.1f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Model Training (Random Forest Classifier)
# We train a Random Forest Classifier — an ensemble of decision trees
# that votes on the class label (Approved/Denied) for each application.
#
# Why Random Forest for Classification?
#   • Naturally handles mixed numeric features without scaling
#   • Reduces overfitting through averaging many trees
#   • Produces reliable probability estimates
#   • Provides built-in feature importance scores
# ─────────────────────────────────────────────────────────────────────────────

classifier = RandomForestClassifier(
    n_estimators=200,    # 200 decision trees in the ensemble
    max_depth=None,      # Trees grow until all leaves are pure
    random_state=42,     # Fixed seed for reproducibility
    n_jobs=-1            # Parallelise training across all CPU cores
)

print("⏳ Training Random Forest Classifier...")
classifier.fit(X_train, y_train)
print("✅ Model training complete!")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Model Evaluation
# We evaluate the classifier using:
#   • Accuracy        — overall correct prediction rate
#   • Precision       — of predicted Approved, how many were truly Approved
#   • Recall          — of actual Approved, how many did we catch
#   • F1-Score        — harmonic mean of precision and recall
#   • Confusion Matrix — True/False Positive & Negative breakdown
# ─────────────────────────────────────────────────────────────────────────────

# Generate class predictions on the test set
y_pred = classifier.predict(X_test)

# Calculate overall accuracy
accuracy = accuracy_score(y_test, y_pred)

print("═" * 55)
print("       📊 MODEL EVALUATION RESULTS")
print("═" * 55)
print(f"  Overall Accuracy : {accuracy * 100:.2f}%")
print("═" * 55)
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Denied (0)', 'Approved (1)']))

# Simple performance interpretation
if accuracy >= 0.90:
    print("✅ Excellent model accuracy! (≥ 90%)")
elif accuracy >= 0.80:
    print("✅ Good model accuracy! (≥ 80%)")
elif accuracy >= 0.70:
    print("⚠️  Moderate model accuracy.")
else:
    print("❌ Weak accuracy — consider feature engineering.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Confusion Matrix Visualization
# The confusion matrix shows how many applications were correctly and
# incorrectly classified in each category.
#
#            Predicted Denied  |  Predicted Approved
#  Actual Denied   [TN]        |  [FP]  (False Positive)
#  Actual Approved [FN]        |  [TP]  (True Positive)
# ─────────────────────────────────────────────────────────────────────────────

cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Performance — Loan Approval Classifier', fontsize=14, fontweight='bold')

# Plot 1: Confusion Matrix heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=axes[0],
    xticklabels=['Denied', 'Approved'],
    yticklabels=['Denied', 'Approved'],
    linewidths=1
)
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('Actual Label')
axes[0].set_title('Confusion Matrix')

# Plot 2: Prediction probability distribution
proba = classifier.predict_proba(X_test)[:, 1]   # Probability of Approved
axes[1].hist(proba[y_test == 1], bins=30, alpha=0.7, color='#2ecc71', label='Actual: Approved')
axes[1].hist(proba[y_test == 0], bins=30, alpha=0.7, color='#e74c3c', label='Actual: Denied')
axes[1].axvline(0.5, color='black', linestyle='--', lw=1.5, label='Threshold = 0.5')
axes[1].set_xlabel('Predicted Probability of Approval')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Prediction Probability Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_performance_loan_approver.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Performance plots saved as model_performance_loan_approver.png")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Feature Importance
# We visualize which features the Random Forest found most useful
# when deciding whether to approve or deny a loan application.
# ─────────────────────────────────────────────────────────────────────────────

# Extract feature importances from trained classifier
importances = classifier.feature_importances_
feature_names = X.columns.tolist()

# Sort by importance descending
sorted_idx = np.argsort(importances)[::-1]

# Bar chart of feature importances
plt.figure(figsize=(9, 5))
bar_colors = ['#2c7bb6' if i == 0 else '#74add1' for i in range(len(feature_names))]
plt.bar(
    [feature_names[i] for i in sorted_idx],
    [importances[i] for i in sorted_idx],
    color=bar_colors
)
plt.title('Feature Importance — Loan Approval Model', fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Importance Score')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('feature_importance_loan_approver.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Feature Importance Ranking:")
for i in sorted_idx:
    print(f"   {feature_names[i]:<18} : {importances[i]:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — Save the Trained Model
# We save the trained classifier to disk using joblib.
# The saved model can be loaded by the Django web interface to make
# real-time predictions without retraining each time.
# ─────────────────────────────────────────────────────────────────────────────

import os

# Save the trained classification model
model_path = 'loan_approver_model.pkl'
joblib.dump(classifier, model_path)
print(f"✅ Model saved to: {os.path.abspath(model_path)}")

# Also save the feature column order so Django uses the correct input shape
feature_order_path = 'loan_features.pkl'
joblib.dump(list(X.columns), feature_order_path)
print(f"✅ Feature order saved to: {os.path.abspath(feature_order_path)}")

print("\n🎉 Loan Approval Classification Model is ready for deployment!")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — Quick Prediction Demo
# We test the saved model with a sample loan application to verify it
# works correctly. This mirrors what the Django interface will do when
# a user submits a loan application form.
# ─────────────────────────────────────────────────────────────────────────────

# Load the saved model
loaded_classifier = joblib.load('loan_approver_model.pkl')

# ── Sample loan application ───────────────────────────────────────────────────
sample_application = pd.DataFrame({
    'income'         : [85000],    # Annual income
    'credit_score'   : [720],      # Credit score (300–850)
    'loan_amount'    : [25000],    # Loan requested
    'years_employed' : [5],        # Years at current job
    'points'         : [75]        # Loyalty/reward points
})

# Predict class and probability
prediction   = loaded_classifier.predict(sample_application)[0]
probability  = loaded_classifier.predict_proba(sample_application)[0]

result_label = '✅ APPROVED' if prediction == 1 else '❌ DENIED'

print("🏦 Loan Application Demo")
print("─" * 40)
print(f"  Income          : ${sample_application['income'][0]:,}")
print(f"  Credit Score    : {sample_application['credit_score'][0]}")
print(f"  Loan Amount     : ${sample_application['loan_amount'][0]:,}")
print(f"  Years Employed  : {sample_application['years_employed'][0]}")
print(f"  Points          : {sample_application['points'][0]}")
print("─" * 40)
print(f"  Decision        : {result_label}")
print(f"  Confidence      : {max(probability)*100:.1f}%")
print(f"  P(Denied)       : {probability[0]*100:.1f}%")
print(f"  P(Approved)     : {probability[1]*100:.1f}%")